# VULGARIS v0.7.0 — Industrial Streaming AI Demo
**Kaggle free CPU · ~25 min · `pip install vulgaris`**

> VULGARIS is not designed to beat DLinear on batch forecasting.
> It is designed for **streaming industrial sensors** where DLinear cannot operate at all.

This notebook demonstrates what VULGARIS uniquely provides:
1. Competitive forecasting accuracy (vs DLinear, same compute)
2. **Correct O(1) memory** — persistent state stays flat while LSTM context grows
3. **Regime detection** — unsupervised operating-mode discovery (DLinear cannot do this)
4. **Causal graph** — which sensors drive OT (DLinear cannot do this)

In [ ]:
!pip install vulgaris plotly -q

In [ ]:
import plotly.io as pio
pio.renderers.default = 'notebook'

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time, warnings
warnings.filterwarnings("ignore")

import vulgaris
from vulgaris import Vulgaris, ModelConfig, Tensor
from vulgaris import SpectralAdamW, CosineSchedule, VulgarisLoss, TrainingPipeline
from vulgaris.config import ASEConfig, SSSRConfig, CRGConfig, RMCConfig, TrainingConfig, HMBConfig

print("VULGARIS", vulgaris.__version__)

## 1. Load ETTh1

In [ ]:
URL = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
try:
    df = pd.read_csv(URL, parse_dates=["date"])
    print("ETTh1:", df.shape)
except Exception:
    n  = 17420
    t  = np.linspace(0, n/24, n)
    df = pd.DataFrame({
        "date": pd.date_range("2016-07-01", periods=n, freq="1h"),
        "HUFL": 10+5*np.sin(2*np.pi*t/24)+np.random.randn(n)*.5,
        "HULL":  6+3*np.sin(2*np.pi*t/24+1)+np.random.randn(n)*.3,
        "MUFL":  3+1.5*np.sin(2*np.pi*t/12)+np.random.randn(n)*.2,
        "MULL":  2+np.cos(2*np.pi*t/24)+np.random.randn(n)*.2,
        "LUFL":  1+.5*np.sin(2*np.pi*t/6)+np.random.randn(n)*.1,
        "LULL":  .5+.3*np.cos(2*np.pi*t/12)+np.random.randn(n)*.1,
        "OT":   25+8*np.sin(2*np.pi*t/24+.5)+np.random.randn(n)*1.,
    })
    print("Synthetic ETTh1-style data")

FEATURES = ["HUFL","HULL","MUFL","MULL","LUFL","LULL","OT"]
C = len(FEATURES)
N = len(df); n_tr=int(N*.6); n_v=int(N*.2)
data = df[FEATURES].values.astype("float32")
sc   = StandardScaler()
tr   = sc.fit_transform(data[:n_tr])
val  = sc.transform(data[n_tr:n_tr+n_v])
tst  = sc.transform(data[n_tr+n_v:])

SEQ = 96; HORIZONS = [96, 192, 336, 720]

def make_windows(d, seq, h):
    X, y = [], []
    for i in range(len(d)-seq-h+1):
        X.append(d[i:i+seq]); y.append(d[i+seq:i+seq+h,-1])
    return np.array(X,"float32"), np.array(y,"float32")

DS = {}
for h in HORIZONS:
    Xt,yt = make_windows(tr,SEQ,h)
    Xv,yv = make_windows(val,SEQ,h)
    Xe,ye = make_windows(tst,SEQ,h)
    DS[h] = dict(Xt=Xt,yt=yt,Xv=Xv,yv=yv,Xe=Xe,ye=ye)
print({h:DS[h]["Xt"].shape for h in HORIZONS})

## 2. Build VULGARIS

In [ ]:
D = 64

cfg = ModelConfig(
    input_dim=C, output_dim=1, n_classes=0,
    ase      = ASEConfig(n_filters=8, n_scales=4, filter_len=32, latent_dim=D),
    sssr     = SSSRConfig(state_dim=D, n_heads=4, d_inner=D*2),
    crg      = CRGConfig(n_nodes=C, n_lags=3),
    rmc      = RMCConfig(n_experts=4),
    hmb      = HMBConfig(embed_dim=D, compress_dim=D//2),
    training = TrainingConfig(lr=3e-4, batch_size=32, seq_len=SEQ,
                              warmup_steps=200, max_steps=10000,
                              gamma_crg=0.0001, grad_clip=1.0),
)
cfg.forecast_horizons = HORIZONS

model  = Vulgaris(cfg)
opt    = SpectralAdamW(model.parameters(), lr=3e-4)
sched  = CosineSchedule(opt, warmup_steps=200, max_steps=10000, min_lr=1e-5)
loss_f = VulgarisLoss(cfg)
pipe   = TrainingPipeline(model, cfg, loss_f, opt, sched)

n_params = sum(p.data.size for p in model.parameters())
print(f"Parameters: {n_params:,}  (d_model={D})")

## 3. Train

In [ ]:
Xt, yt = DS[96]["Xt"], DS[96]["yt"]
B, EPOCHS = 32, 20
t0, log   = time.time(), []

model.train()
for ep in range(EPOCHS):
    idx = np.random.permutation(len(Xt))
    el  = 0; nb = 0
    for s in range(0, len(Xt)-B, B):
        xb = Xt[idx[s:s+B]].transpose(0,2,1)
        yb = yt[idx[s:s+B], 0:1]
        m  = pipe.train_step(xb, yb)
        el += m.get("total_loss", 0); nb += 1
    log.append(el/max(nb,1))
    if (ep+1)%5==0 or ep==0:
        print(f"Epoch {ep+1:2d}  loss={log[-1]:.4f}  {time.time()-t0:.0f}s")

model.eval()

go.Figure(go.Scatter(y=log, mode="lines+markers",
    line=dict(color="#00D4FF",width=2))).update_layout(
    template="plotly_dark", height=280,
    title="Training loss (total_loss)",
    xaxis_title="Epoch", yaxis_title="Loss").show()

## 4. DLinear Baseline

In [ ]:
class DLinear:
    def fit(self, X, y, epochs=50, lr=5e-4, B=64):
        N,T,C = X.shape
        Xf = X.reshape(N,-1).astype("float64"); yf=y.astype("float64")
        self.W = np.zeros((T*C, y.shape[1])); self.b = np.zeros(y.shape[1])
        for _ in range(epochs):
            i = np.random.permutation(N)
            for s in range(0,N-B,B):
                xb,yb = Xf[i[s:s+B]], yf[i[s:s+B]]
                e = xb@self.W+self.b-yb
                self.W -= lr*(xb.T@e)/B; self.b -= lr*e.mean(0)
    def predict(self,X):
        return (X.reshape(len(X),-1).astype("float64")@self.W+self.b).astype("float32")

dl={}
for h in HORIZONS:
    m=DLinear(); m.fit(DS[h]["Xt"],DS[h]["yt"]); dl[h]=m
    print(f"DLinear H={h} done")

## 5. Forecasting — Normalised Scale

In [ ]:
CHUNK=64; results={}
for h in HORIZONS:
    Xe,ye = DS[h]["Xe"],DS[h]["ye"]
    vp = np.concatenate([
        model(Tensor(Xe[s:s+CHUNK].transpose(0,2,1)))[0].data[:,0]
        for s in range(0,len(Xe)-CHUNK,CHUNK)
    ])
    dp = dl[h].predict(Xe)[:,0]
    n  = min(len(vp),len(dp),len(ye))
    vt = ye[:n,0]
    results[h] = {"v_mae": mean_absolute_error(vt,vp[:n]),
                  "d_mae": mean_absolute_error(vt,dp[:n]),
                  "vp":vp[:300],"dp":dp[:300],"true":vt[:300]}
    print(f"H={h:4d} | VULGARIS {results[h]['v_mae']:.4f} | DLinear {results[h]['d_mae']:.4f}")

print("\nContext: DLinear is a linear model that is extremely hard to beat on pure")
print("batch forecasting. VULGARIS's advantage is what it does WHILE forecasting.")

In [ ]:
fig = make_subplots(2,2,subplot_titles=[f"H={h}" for h in HORIZONS])
for i,h in enumerate(HORIZONS):
    r,c = i//2+1,i%2+1
    x=list(range(300))
    fig.add_trace(go.Scatter(x=x,y=results[h]["true"].tolist(),name="Truth",
        line=dict(color="white",width=1.5),showlegend=(i==0)),row=r,col=c)
    fig.add_trace(go.Scatter(x=x,y=results[h]["vp"].tolist(),name="VULGARIS",
        line=dict(color="#00D4FF",width=2),showlegend=(i==0)),row=r,col=c)
    fig.add_trace(go.Scatter(x=x,y=results[h]["dp"].tolist(),name="DLinear",
        line=dict(color="#FF6B6B",width=1.5,dash="dash"),showlegend=(i==0)),row=r,col=c)
fig.update_layout(template="plotly_dark",height=600,
    title="ETTh1 — Step-1 Forecast (normalised)").show()

## 6. O(1) Streaming Memory — Correct Measurement

The O(1) claim is about **persistent state size** (what you must keep in RAM between steps).
- VULGARIS: hidden state is a fixed-size vector regardless of how many steps have passed
- An LSTM that caches past hidden states for attention/context: grows linearly with T

We measure **state size** — not peak RAM during a forward pass (which includes model weights for both).

In [ ]:
# VULGARIS persistent state = SSR hidden states + HTD states
# Measure it directly
state = model.init_state(batch_size=1)
v_state_mb = sum(
    np.asarray(s.data).nbytes for s in state.sssr_states
    if hasattr(s, "data")
) / 1024 / 1024
# Add HTD states
v_state_mb += sum(
    np.asarray(s.data).nbytes for s in state.htd_states
    if hasattr(s, "data")
) / 1024 / 1024
print(f"VULGARIS persistent state: {v_state_mb:.4f} MB  (constant for any T)")

# LSTM context cache: stores one hidden vector per past step
hidden = 64
T_vals = [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000]
lstm_context_mb = [T * hidden * 4 / 1024 / 1024 for T in T_vals]   # float32

print(f"\nLSTM context cache (h_dim={hidden}) at different T values:")
for T, mb in zip(T_vals, lstm_context_mb):
    print(f"  T={T:>8,}  →  {mb:.3f} MB")

In [ ]:
fig = go.Figure()

# VULGARIS: flat line at actual state size
fig.add_trace(go.Scatter(
    x=T_vals, y=[v_state_mb]*len(T_vals),
    name=f"VULGARIS persistent state ({v_state_mb:.3f} MB)",
    line=dict(color="#00D4FF", width=4),
    fill="tozeroy", fillcolor="rgba(0,212,255,0.15)"
))

# LSTM context cache: grows
fig.add_trace(go.Scatter(
    x=T_vals, y=lstm_context_mb,
    name="LSTM context cache (grows with T)",
    line=dict(color="#FF6B6B", width=3, dash="dash"),
    fill="tozeroy", fillcolor="rgba(255,107,107,0.1)"
))

fig.update_layout(
    template="plotly_dark", height=450,
    title="<b>Persistent State Size: VULGARIS O(1) vs LSTM O(T)</b>",
    xaxis_title="Steps processed (T)", yaxis_title="State memory (MB)",
    xaxis_type="log",
    annotations=[
        dict(x=4, y=v_state_mb*8,
             text=f"<b>VULGARIS: {v_state_mb:.3f} MB forever</b>",
             showarrow=True, arrowcolor="#00D4FF",
             font=dict(color="#00D4FF", size=13)),
        dict(x=4, y=lstm_context_mb[4]*0.6,
             text="<b>LSTM: grows without bound</b>",
             showarrow=True, arrowcolor="#FF6B6B",
             font=dict(color="#FF6B6B", size=13)),
    ]
)
fig.show()

print(f"\nAt T=1,000,000 steps:")
print(f"  VULGARIS state: {v_state_mb:.4f} MB")
print(f"  LSTM context  : {lstm_context_mb[-1]:.1f} MB")
print(f"  Ratio         : {lstm_context_mb[-1]/max(v_state_mb,0.001):.0f}x")

## 7. Regime Discovery — DLinear Cannot Do This

VULGARIS's RMC (Regime Mixture Core) automatically detects operating modes
from unlabelled data. ETTh1 has daily and seasonal patterns — the model learns
to distinguish them without any labels.

In [ ]:
rw_list=[]; CHUNK=32; Xe96=DS[96]["Xe"]
for s in range(0, min(600,len(Xe96))-CHUNK, CHUNK):
    xb=Tensor(Xe96[s:s+CHUNK].transpose(0,2,1))
    _,aux=model(xb)
    rw=aux.get("regime_weights")
    if rw is not None:
        rw_list.extend([rw]*CHUNK)

if rw_list:
    hard = np.array([r.argmax() for r in rw_list[:600]])
    ot   = Xe96[:len(hard),-1,-1]
    dates= df["date"].values[n_tr+n_v:n_tr+n_v+len(hard)]

    COLS=["#00D4FF","#FF6B6B","#51CF66","#FFD43B"]
    fig=make_subplots(2,1,
        subplot_titles=["OT signal coloured by detected regime",
                        "Regime assignment over time"],
        row_heights=[0.65,0.35])
    for k in range(4):
        mask=hard==k
        if mask.any():
            fig.add_trace(go.Scatter(
                x=list(range(mask.sum())),
                y=ot[mask].tolist(), mode="markers",
                marker=dict(color=COLS[k],size=3,opacity=0.8),
                name=f"Regime {k}"), row=1,col=1)
    fig.add_trace(go.Scatter(y=hard.tolist(), mode="lines",
        line=dict(color="white",width=1),showlegend=False),row=2,col=1)
    fig.update_layout(template="plotly_dark",height=520,
        title="<b>RMC Unsupervised Regime Detection on ETTh1</b>")
    fig.show()

    # Regime balance pie
    counts=np.bincount(hard,minlength=4)
    go.Figure(go.Pie(labels=[f"Regime {k}" for k in range(4)],
        values=counts.tolist(),
        marker=dict(colors=COLS), hole=0.35)).update_layout(
        template="plotly_dark",height=320,
        title="Expert utilisation — load balance").show()

print("DLinear: produces a single linear forecast. Cannot detect regimes.")
print("VULGARIS: simultaneously forecasts AND discovers operating modes.")

## 8. Causal Graph — Which Sensors Drive OT?

VULGARIS learns a sparse DAG over the 7 ETTh1 channels. After training, we can ask:
*which sensors causally precede OT (the target)?*

DLinear uses all inputs with equal weight. VULGARIS learns the causal structure.

In [ ]:
# Extract learned adjacency matrix from CRG
W = model.crg.W.data.copy()   # (n_nodes, n_nodes)
# Apply mask if available
if hasattr(model.crg, "M"):
    import scipy.special
    mask = 1 / (1 + np.exp(-model.crg.M.data))
    W_eff = W * mask
else:
    W_eff = W

# Heatmap of causal strengths
fig=go.Figure(go.Heatmap(
    z=np.abs(W_eff).tolist(),
    x=FEATURES, y=FEATURES,
    colorscale="Blues",
    colorbar=dict(title="Edge strength")
))
fig.update_layout(template="plotly_dark",height=420,
    title="<b>Learned Causal Graph — |W_eff| edge strengths</b>",
    xaxis_title="Effect (column)", yaxis_title="Cause (row)")
fig.show()

# Which sensors most strongly cause OT (last column)?
ot_col = len(FEATURES)-1
causes = [(FEATURES[i], float(abs(W_eff[i,ot_col])))
          for i in range(len(FEATURES)) if i != ot_col]
causes.sort(key=lambda x: -x[1])

print("Causal strength → OT:")
for name, strength in causes:
    bar = "█" * int(strength*50)
    print(f"  {name:6s}: {bar} {strength:.4f}")
print("\nDLinear has NO notion of causal direction between sensors.")

## 9. Summary Dashboard

In [ ]:
# ── MAE results table ──────────────────────────────────────────────────
rows_v = [f"{results[h]['v_mae']:.4f}" for h in HORIZONS]
rows_d = [f"{results[h]['d_mae']:.4f}" for h in HORIZONS]
rows_g = [f"+{results[h]['v_mae']-results[h]['d_mae']:.4f}" for h in HORIZONS]
winner = ["✓ VULGARIS" if results[h]["v_mae"]<=results[h]["d_mae"]
          else "✓ DLinear" for h in HORIZONS]

fig_tbl = go.Figure(go.Table(
    header=dict(
        values=["Horizon","VULGARIS MAE","DLinear MAE","Gap","Winner"],
        fill_color="#1A2438", font=dict(color="white",size=13),
        align="center", height=32),
    cells=dict(
        values=[[str(h) for h in HORIZONS], rows_v, rows_d, rows_g, winner],
        fill_color=[["#0D1B2A"]*4],
        font=dict(color=["white","#00D4FF","#FF6B6B","#aaa","#51CF66"], size=12),
        align="center", height=28)
))
fig_tbl.update_layout(template="plotly_dark", height=230,
    title=f"<b>ETTh1 Forecasting Results — VULGARIS v{vulgaris.__version__}</b>")
fig_tbl.show()

# ── Capability comparison ───────────────────────────────────────────────────
caps = ["Forecasting","O(1) streaming","Regime detection",
        "Causal graph","TTT adaptation","Safety (CBF)","Multi-rate sensors"]
v_vals = [0.6, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]   # 0=no, 1=yes, 0.6=fair
d_vals = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

fig_cap = go.Figure()
fig_cap.add_trace(go.Bar(name="VULGARIS", y=caps, x=v_vals,
    orientation="h", marker_color="#00D4FF", opacity=0.85))
fig_cap.add_trace(go.Bar(name="DLinear", y=caps, x=d_vals,
    orientation="h", marker_color="#FF6B6B", opacity=0.85))
fig_cap.update_layout(
    template="plotly_dark", barmode="group", height=380,
    title="<b>Capability Comparison</b>",
    xaxis=dict(tickvals=[0,0.5,1], ticktext=["No","Partial","Yes"],
               range=[0,1.2]),
    legend=dict(orientation="h", y=1.08))
fig_cap.show()